**Alunos: Gabriel Rosa Galdino, Rodrigo Almeida Piropo**

**Matrícula: 202320098, 202320103**

# Segunda Avaliação - Algoritmo Genético

### 1. Definição do Problema e Funções Auxiliares

Nesta seção, definimos a estrutura básica do problema das 8 Rainhas.Onde cada indivíduo da população é representado por uma sequência de números (alfabeto finito), onde cada número indica a linha em que a rainha está posicionada na respectiva coluna.

As funções principais aqui são:
* **`Problem`**: Classe base para compatibilidade com o codigo original.
* **`eight_queens_heuristic`**: Calcula o número de pares de rainhas se atacando.
* **`fitness_8queens`**: Função de aptidão. Como o objetivo do AG é maximizar o desempenho, invertemos a heurística. O valor máximo ideal é 28 (nenhum ataque).

In [11]:
import numpy as np
import time
import matplotlib.pyplot as plt

# --- Definição do Problema ---

class Problem:
    def __init__(self, states, initial, goal, actions, transition_model, cost):
        self.states = states
        if initial not in states:
            self.states.append(initial)
        self.initial = initial
        if goal not in states:
            self.states.append(goal)
        self.goal = goal
        self.actions = actions
        self.transition_model = transition_model
        self.cost = cost
    
    def get_actions(self, state): return self.actions[state]
    def result(self, state, action): return self.transition_model[state][action]
    def goal_test(self, state): return state == self.goal
    def action_cost(self, state1, action, state2): return 1

# --- Funções Específicas das 8 Rainhas ---

def eight_queens_heuristic(state):
    """
    Calcula o número de pares de rainhas se atacando.
    O ideal é 0.
    """
    h = 0
    for i in range(8):
        for j in range(i+1, 8):
            if state[i] == state[j]: # Mesma linha
                h += 1
                continue
            if state[i] == state[j] + (j - i): # Diagonal
                h += 1
                continue
            if state[i] == state[j] - (j - i): # Diagonal
                h += 1
    return h

def fitness_8queens(state):
    """
    Função de Aptidão (Fitness).
    O número máximo de pares não-atacantes é 28 (8 escolhe 2).
    Fitness = 28 - h.
    O objetivo é maximizar o fitness (max = 28).
    """
    return 28 - eight_queens_heuristic(state)

# Estado inicial aleatório para teste
state_initial = np.random.randint(1, 9, 8)
print("Estado inicial teste:", state_initial)
print("Heurística (ataques):", eight_queens_heuristic(state_initial))
print("Fitness (não-ataques):", fitness_8queens(state_initial))

Estado inicial teste: [7 8 2 1 6 3 6 4]
Heurística (ataques): 5
Fitness (não-ataques): 23


### 2. Implementação do Algoritmo Genético Avançado

Esta classe implementa o Algoritmo Genético com todas as variações solicitadas na atividade. O código permite testes para as seguintes estratégias:

1.  **Tamanho da População (`population_size`)**: Permite definir o tamanho da geração inicial e subsequentes, passando o tamanho desejado como parâmetro.
2.  **Número de Mistura (`rho`)**:
    * **$\rho=1$ (Reprodução Assexuada)**: O indivíduo é clonado e a variabilidade ocorre apenas via mutação, simulando uma busca estocástica de feixe.
    * **$\rho=2$ (Padrão)**: Cruzamento de dois pais com um ponto de corte.
    * **$\rho=3$ (Multiparental)**: Implementação de cruzamento com 3 pais, utilizando múltiplos pontos de corte para combinar genes de três fontes.
3.  **Seleção (`selection_strategy`)**: Alternância entre o método de "Roleta" (proporcional ao fitness) e "Torneio" (selecionar os melhores de um subgrupo aleatório).
4.  **Montagem da Próxima Geração**:
    * **Elitismo**: Preserva os melhores indivíduos da geração anterior para garantir que a aptidão não diminua.
    * **Abate (Culling)**: Descarta os indivíduos com pior desempenho antes da reprodução (abaixo de um limiar) para acelerar a convergência.

In [12]:
class GeneticAlgorithmAdvanced:
    def __init__(self, 
                 population_size=100, 
                 rho=2, 
                 selection_strategy='roulette', 
                 recombination_strategy='single_point', 
                 mutation_rate=0.1, 
                 elitism=False, 
                 culling=False,
                 max_generations=1000):
        
        self.pop_size = population_size
        self.rho = rho
        self.selection_strategy = selection_strategy
        self.recombination_strategy = recombination_strategy
        self.mutation_rate = mutation_rate
        self.elitism = elitism
        self.culling = culling
        self.max_generations = max_generations
        self.n_queens = 8

    def generate_population(self):
        """Gera uma população inicial aleatória."""
        return [np.random.randint(1, 9, 8) for _ in range(self.pop_size)]

    def select_parents(self, population, fitnesses, n_parents):
        """Seleciona pais baseados na estratégia definida."""
        if self.selection_strategy == 'roulette':
            # Roleta viciada (proporcional ao fitness)
            total_fit = np.sum(fitnesses)
            if total_fit == 0:
                probs = np.ones(len(fitnesses)) / len(fitnesses)
            else:
                probs = fitnesses / total_fit
            indices = np.random.choice(len(population), n_parents, p=probs)
            return [population[i] for i in indices]
        
        elif self.selection_strategy == 'tournament':
            # Torneio: Pega k indivíduos aleatórios e escolhe o melhor
            k = 3 # Tamanho do torneio
            parents = []
            for _ in range(n_parents):
                candidates_idx = np.random.choice(len(population), k)
                best_idx = candidates_idx[np.argmax([fitnesses[i] for i in candidates_idx])]
                parents.append(population[best_idx])
            return parents
        
        return population[:n_parents] # Fallback

    def reproduce(self, parents):
        """
        Realiza o cruzamento (crossover) baseado no número de pais (rho).
        """
        n = self.n_queens
        
        # Caso Rho = 1 (Reprodução Assexuada)
        if self.rho == 1:
            # A "recombinação" de 1 pai é apenas a cópia dele mesmo.
            # A variabilidade virá apenas da mutação.
            return parents[0].copy()

        # Caso Rho = 2 (Padrão)
        elif self.rho == 2:
            p1, p2 = parents
            if self.recombination_strategy == 'single_point':
                c = np.random.randint(1, n)
                return np.array(list(p1[:c]) + list(p2[c:]))
            elif self.recombination_strategy == 'two_point':
                c1 = np.random.randint(1, n-1)
                c2 = np.random.randint(c1+1, n)
                return np.array(list(p1[:c1]) + list(p2[c1:c2]) + list(p1[c2:]))

        # Caso Rho = 3 (3 Pais)
        elif self.rho == 3:
            p1, p2, p3 = parents
            # Estratégia: 2 pontos de corte dividindo o genoma em 3 partes
            # Filho recebe: Começo do P1, Meio do P2, Fim do P3
            c1 = np.random.randint(1, n-1)
            c2 = np.random.randint(c1+1, n)
            return np.array(list(p1[:c1]) + list(p2[c1:c2]) + list(p3[c2:]))
        
        # Padrão fallback
        return parents[0].copy()

    def mutate(self, individual):
        """Aplica mutação com base na taxa definida."""
        if np.random.random() < self.mutation_rate:
            n = len(individual)
            c = np.random.randint(0, n)
            individual[c] = np.random.randint(1, 9)
        return individual

    def search(self):
        """Executa o Algoritmo Genético."""
        population = self.generate_population()
        
        for generation in range(self.max_generations):
            # Calcula fitness de toda a população
            fitnesses = np.array([fitness_8queens(ind) for ind in population])
            
            # Verifica se encontrou solução (Fitness 28)
            best_idx = np.argmax(fitnesses)
            if fitnesses[best_idx] == 28:
                return population[best_idx], generation
            
            new_population = []
            
            # --- Estratégias de Próxima Geração ---
            
            # 1. Elitismo: Mantém os melhores da geração anterior
            if self.elitism:
                # Ordena indices pelo fitness (descendente)
                sorted_indices = np.argsort(fitnesses)[::-1]
                n_elites = max(1, int(self.pop_size * 0.05)) # 5% de elite
                for i in range(n_elites):
                    new_population.append(population[sorted_indices[i]])
            
            # 2. Abate (Culling): Remove os piores antes da seleção
            # (Aqui implementado simplificadamente filtrando a população base de seleção)
            if self.culling:
                # Remove os 20% piores
                limit = int(self.pop_size * 0.8)
                sorted_indices = np.argsort(fitnesses)[::-1]
                valid_indices = sorted_indices[:limit]
                pop_for_selection = [population[i] for i in valid_indices]
                fit_for_selection = fitnesses[valid_indices]
            else:
                pop_for_selection = population
                fit_for_selection = fitnesses

            # Preenche o restante da nova população
            while len(new_population) < self.pop_size:
                # Seleciona pais
                parents = self.select_parents(pop_for_selection, fit_for_selection, self.rho)
                
                # Recombina (Crossover)
                child = self.reproduce(parents)
                
                # Muta
                child = self.mutate(child)
                
                new_population.append(child)
            
            population = new_population

        # Se acabar o tempo, retorna o melhor encontrado
        fitnesses = np.array([fitness_8queens(ind) for ind in population])
        best_idx = np.argmax(fitnesses)
        return population[best_idx], self.max_generations

# Teste rápido
ga = GeneticAlgorithmAdvanced(population_size=50, rho=2, mutation_rate=0.1)
solucao, passos = ga.search()
print(f"Solução encontrada em {passos} gerações.")
print("Estado:", solucao)
print("Heurística:", eight_queens_heuristic(solucao))


Solução encontrada em 374 gerações.
Estado: [5 7 2 6 3 1 8 4]
Heurística: 0


### 3. Função para Executar os Experimentos

Para atender ao requisito do exercício de encontrar a melhor combinação de parâmetros, criamos a função `run_experiment`.

* O algoritmo é executado **100 vezes** para cada configuração (parâmetro padrão da função).
* São coletadas as taxas de sucesso (quantas vezes encontrou a solução ótima) e o número médio de gerações/passos.
* Isso permite avaliar tanto a eficiência (tempo/passos) quanto a eficácia (taxa de acerto).

In [13]:
def run_experiment(description, params, repetitions=100):
    """
    Roda o experimento 100 vezes e coleta métricas.
    """
    print(f"--- Iniciando Experimento: {description} ---")
    print(f"Parâmetros: {params}")
    
    success_count = 0
    total_steps = 0
    start_time = time.time()
    
    for _ in range(repetitions):
        ga = GeneticAlgorithmAdvanced(**params)
        sol, steps = ga.search()
        
        # Verifica se achou solução ótima (Heuristica 0 / Fitness 28)
        if eight_queens_heuristic(sol) == 0:
            success_count += 1
            total_steps += steps
        else:
            # Se falhou, computamos o máximo de passos (penalidade) ou ignoramos na média?
            total_steps += steps

    avg_steps = total_steps / repetitions
    elapsed = time.time() - start_time
    
    print(f"Taxa de Sucesso: {success_count}/{repetitions} ({success_count}%)")
    print(f"Média de Gerações: {avg_steps:.2f}")
    print(f"Tempo Total: {elapsed:.2f}s")
    print("-" * 30 + "\n")
    
    return {
        'description': description,
        'success_rate': success_count,
        'avg_steps': avg_steps
    }

### 4. Execução dos Testes 

Abaixo realizamos a bateria de testes sugerida na atividade. Os testes estão divididos em:

1.  **Tamanho da População**: Variação entre populações pequenas (20) e padrão (100).
2.  **Valor de Mistura ($\rho$)**: Teste crítico valendo **ponto extra**. Comparação entre $\rho=1$ (Busca de Feixe), $\rho=2$ (Clássico) e $\rho=3$ (Genética complexa).
3.  **Estratégias de Seleção**: Comparativo entre Roleta e Torneio.
4.  **Taxa de Mutação**: Impacto de taxas baixas (5%) vs altas (20%) na diversidade.
5.  **Estratégias de Sobrevivência**: Testes com Elitismo e Abate (Culling) ativados.

**Nota sobre Pontos Extras:**
* Para o segundo ponto extra (1000 repetições nos 3 melhores resultados), você pode rodar novamente a função `run_experiment` apenas para os vencedores alterando o parâmetro `repetitions=1000`.

Ao final, um resumo tabular apresenta a taxa de acerto e o custo computacional de cada abordagem.

In [14]:
results = []

# Configuração Padrão (Baseline)
base_params = {
    'population_size': 100,
    'rho': 2,
    'selection_strategy': 'roulette',
    'mutation_rate': 0.1,
    'elitism': False,
    'culling': False
}

# 1. Teste de Tamanho de População
results.append(run_experiment("População Pequena (20)", {**base_params, 'population_size': 20}))
results.append(run_experiment("População Padrão (100)", {**base_params, 'population_size': 100}))
# results.append(run_experiment("População Grande (200)", {**base_params, 'population_size': 200}))

# 2. Teste do Valor de Mistura (Rho) 
# Rho = 1 (Assexuado/Feixe Estocástico), Rho = 2 (Padrão), Rho = 3 (3 Pais)
results.append(run_experiment("Rho=1 (Assexuado)", {**base_params, 'rho': 1}))
results.append(run_experiment("Rho=3 (3 Pais)", {**base_params, 'rho': 3}))

# 3. Teste de Estratégias de Seleção
# Roleta (já testado no padrão) vs Torneio
results.append(run_experiment("Seleção Torneio", {**base_params, 'selection_strategy': 'tournament'}))

# 4. Teste de Taxas de Mutação
results.append(run_experiment("Mutação Baixa (5%)", {**base_params, 'mutation_rate': 0.05}))
results.append(run_experiment("Mutação Alta (20%)", {**base_params, 'mutation_rate': 0.20}))

# 5. Teste de Montagem da Próxima Geração (Elitismo e Abate)
results.append(run_experiment("Com Elitismo", {**base_params, 'elitism': True}))
results.append(run_experiment("Com Abate (Culling)", {**base_params, 'culling': True}))

# Resumo Final
print("\n=== RESUMO DOS RESULTADOS ===")
print(f"{'Descrição':<30} | {'Sucesso (%)':<12} | {'Média Gerações':<15}")
print("-" * 65)
for r in results:
    print(f"{r['description']:<30} | {r['success_rate']:<12} | {r['avg_steps']:<15.2f}")

--- Iniciando Experimento: População Pequena (20) ---
Parâmetros: {'population_size': 20, 'rho': 2, 'selection_strategy': 'roulette', 'mutation_rate': 0.1, 'elitism': False, 'culling': False}
Taxa de Sucesso: 28/100 (28%)
Média de Gerações: 877.72
Tempo Total: 82.24s
------------------------------

--- Iniciando Experimento: População Padrão (100) ---
Parâmetros: {'population_size': 100, 'rho': 2, 'selection_strategy': 'roulette', 'mutation_rate': 0.1, 'elitism': False, 'culling': False}
Taxa de Sucesso: 88/100 (88%)
Média de Gerações: 369.45
Tempo Total: 170.12s
------------------------------

--- Iniciando Experimento: Rho=1 (Assexuado) ---
Parâmetros: {'population_size': 100, 'rho': 1, 'selection_strategy': 'roulette', 'mutation_rate': 0.1, 'elitism': False, 'culling': False}
Taxa de Sucesso: 70/100 (70%)
Média de Gerações: 583.76
Tempo Total: 218.54s
------------------------------

--- Iniciando Experimento: Rho=3 (3 Pais) ---
Parâmetros: {'population_size': 100, 'rho': 3, 'selecti

# Informações sobre as Implementações Especiais 

- Rho = 1 (Reprodução Assexuada): Como solicitado uma proposta, implementei como uma clonagem seguida de mutação. Isso efetivamente transforma o GA em uma Stochastic Beam Search (Busca de Feixe Estocástica), onde múltiplos estados evoluem independentemente através de mutações, competindo apenas na etapa de seleção.

- Rho = 3 (3 Pais): Implementei um crossover com 2 pontos de corte. O filho herda o primeiro terço do Pai 1, o terço do meio do Pai 2 e o terço final do Pai 3. Isso aumenta a diversidade genética combinando traços de três indivíduos bem-sucedidos.

- Seleção: Implementei o "Torneio" como alternativa à Roleta, pois geralmente preserva melhor a diversidade e evita convergência prematura em populações grandes.